# Dengue e comorbidades — execução no Google Colab

Notebook para executar o pipeline versionado do projeto em uma sessão Colab. No menu **Ambiente de execução → Alterar tipo de ambiente de execução**, selecione GPU para o candidato MLP. O projeto requer Python 3.13; o notebook instala essa versão gerenciada com `uv`, sem depender da versão Python padrão do Colab.

**Atenção:** as buscas podem levar horas. O Optuna persiste no Google Drive; execute e conclua **um candidato por vez** e, se a sessão desconectar, monte o Drive novamente e repita apenas o candidato interrompido. Dados e resultados permanecem no Drive, fora do repositório. As células deste notebook baixam apenas 2022 e 2023. **Não baixe nem avalie 2024 aqui**: o teste é congelado e reservado para a rotina autorizada de avaliação única, depois do congelamento dos modelos.

O repositório precisa conter os arquivos do sistema (incluindo `scripts/build_models.py`). Se você acabou de criá-los localmente, envie as alterações para o GitHub antes de executar a célula de clone.


## 1. Montar o Google Drive

Os microdados, bancos Optuna, rastreamento, relatórios e modelos serão gravados na pasta persistente `MyDrive/dengue-comorbidity-risk`.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/dengue-comorbidity-risk')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
print(f'Resultados persistentes: {DRIVE_PROJECT}')

## 2. Obter o código

A cópia do código fica em `/content` (temporária); não clone o repositório dentro da pasta de microdados.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/PabloThePmX/dengue-comorbidity-risk.git'
WORKSPACE = Path('/content/dengue-comorbidity-risk')
if not WORKSPACE.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(WORKSPACE)],
        check=True,
    )
assert (WORKSPACE / 'scripts' / 'build_models.py').is_file(), (
    'Checkout sem o pipeline. Envie as alterações ao GitHub e reexecute.'
)
print(f'Código: {WORKSPACE}')

## 3. Instalar o ambiente do projeto

Instala Python 3.13.12 isolado e sincroniza as versões definidas no `uv.lock`. As chamadas posteriores usam esse Python, ainda que o kernel Colab use outra versão.

In [ ]:
import shutil
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
if UV is None:
    raise FileNotFoundError('uv não foi instalado ou não está no PATH.')
subprocess.run([UV, 'python', 'install', '3.13.12'], check=True)
subprocess.run(
    [UV, 'sync', '--python', '3.13.12', '--frozen'],
    cwd=WORKSPACE,
    check=True,
)
subprocess.run(
    [
        UV, 'run', '--python', '3.13.12', 'python', '-c',
        'import sys, torch; '
        'print("Python:", sys.version); '
        'print("PyTorch:", torch.__version__); '
        'print("GPU:", torch.cuda.get_device_name(0) '
        'if torch.cuda.is_available() else "indisponível")',
    ],
    cwd=WORKSPACE,
    check=True,
)
print('Selecione runtime GPU antes de treinar o MLP, se ainda não o fez.')

## 4. Configurar caminhos persistentes

O YAML abaixo deriva de `config/base.yaml`, mas aponta dados e saídas para o Drive. A configuração será salva no Drive e pode ser reutilizada ao reconectar.

In [ ]:
import json
import re

DATA_DIR = DRIVE_PROJECT / 'data'
MLRUNS_DIR = DRIVE_PROJECT / 'mlruns'
REPORTS_DIR = DRIVE_PROJECT / 'reports'
MODELS_DIR = DRIVE_PROJECT / 'models'
for directory in (DATA_DIR, MLRUNS_DIR, REPORTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

config_text = (WORKSPACE / 'config' / 'base.yaml').read_text(encoding='utf-8')
for key, value in {
    'dados': DATA_DIR,
    'mlruns': MLRUNS_DIR,
    'relatorios': REPORTS_DIR,
    'modelos': MODELS_DIR,
}.items():
    pattern = rf'(?m)^(  {key}:).*$'
    config_text, replaced = re.subn(
        pattern,
        lambda match, path=value: f'{match.group(1)} {json.dumps(str(path))}',
        config_text,
    )
    if replaced != 1:
        raise ValueError(f'Esperava exatamente um caminho {key!r} em config/base.yaml')

CONFIG_PATH = DRIVE_PROJECT / 'config_colab.yaml'
CONFIG_PATH.write_text(config_text, encoding='utf-8')
print(CONFIG_PATH.read_text(encoding='utf-8'))

## 5. Baixar e preparar somente os dados de desenvolvimento

A função oficial do projeto baixa DBC, converte para Parquet em blocos e atualiza o manifesto com hashes. Esta célula fixa explicitamente os anos **2022 e 2023**; não a altere para incluir 2024.

In [ ]:
fetch_code = '\n'.join([
    'from pathlib import Path',
    'from dengue_sinan.download import fetch_year',
    f'data_dir = Path({str(DATA_DIR)!r})',
    'for year in (2022, 2023):',
    '    entry = fetch_year(year, data_dir)',
    '    print(year, entry["linhas"], entry["sha256_parquet"])',
])
subprocess.run(
    [UV, 'run', '--python', '3.13.12', 'python', '-c', fetch_code],
    cwd=WORKSPACE,
    check=True,
)

In [ ]:
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
for year in ('2022', '2023'):
    if year not in manifest:
        raise ValueError(f'Manifesto sem o ano de desenvolvimento {year}')
    item = manifest[year]
    print(year, 'linhas=', item['linhas'], 'sha256=', item['sha256_parquet'])
print('Manifesto verificado apenas para os anos de desenvolvimento solicitados.')

## 6. Busca e validação de M1

Execute uma célula de candidato por vez. Cada busca tem orçamento configurado de 60–90 minutos e o estudo Optuna fica no Drive. Se houver desconexão, reconecte, execute novamente as células 1–4 e repita apenas a célula do candidato interrompido: o SQLite será retomado. O TabFM não está incluído porque depende da verificação de licença e do model card.

As buscas reproduzem cinco candidatos e produzem OOF. Não avalie o conjunto de teste nesta etapa.

In [ ]:
import subprocess


def run_stage(*arguments):
    command = [
        UV, 'run', '--python', '3.13.12', 'python',
        'scripts/build_models.py', '--config', str(CONFIG_PATH), *arguments,
    ]
    print('Executando:', ' '.join(command))
    subprocess.run(command, cwd=WORKSPACE, check=True)

def tune_candidate(model):
    run_stage('--stage', 'tune', '--model', model)

In [ ]:
tune_candidate('elastic_net')

In [ ]:
tune_candidate('random_forest')

In [ ]:
tune_candidate('xgboost')

In [ ]:
tune_candidate('catboost')

In [ ]:
tune_candidate('mlp')

## 7. Seleção OOF, calibração e retreino final

Rode a seleção somente depois que os cinco arquivos `*_melhor_validacao.json` forem gerados. O retreino ajusta M1/B1 em todo 2022–2023 sem reajustar o calibrador ou o limiar.

In [ ]:
run_stage('--stage', 'select')
print((REPORTS_DIR / 'm1' / 'comparacao_oof.md').read_text(encoding='utf-8'))

In [ ]:
run_stage('--stage', 'train')
print('M1:', MODELS_DIR / 'm1_final.joblib')
print('B1:', MODELS_DIR / 'b1_final.joblib')

## 8. Treinar E1 e gerar relatórios de perfil

Preencha os valores de referência acordados pelo grupo. Eles são usados em RERI/AP e na tabela de perfis; **não foram inventados pelo notebook**. A célula interrompe a execução enquanto houver `None`. Idades precisam corresponder às faixas `0–4`, `5–14`, `15–39`, `40–59`, `60–79`, `80+`.

In [ ]:
E1_REFERENCE_PROFILE = {
    'idade_anos': None,  # preencher idade de referência para RERI/AP
    'sexo': None,        # 'F', 'M' ou 'I'
    'gestante': None,    # True ou False
}
AGE_REFERENCES = {
    '0–4': None,
    '5–14': None,
    '15–39': None,
    '40–59': None,
    '60–79': None,
    '80+': None,
}
GESTANTE_PERFIS = None  # preencher True ou False para a tabela completa de perfis

if any(value is None for value in E1_REFERENCE_PROFILE.values()):
    raise ValueError('Preencha E1_REFERENCE_PROFILE com as decisões do grupo.')
if set(AGE_REFERENCES) != {'0–4', '5–14', '15–39', '40–59', '60–79', '80+'} or any(
    value is None for value in AGE_REFERENCES.values()
):
    raise ValueError('Preencha uma idade de referência para cada faixa etária.')
if GESTANTE_PERFIS is None:
    raise ValueError('Defina GESTANTE_PERFIS como True ou False.')

E1_REFERENCE_PATH = DRIVE_PROJECT / 'e1_reference_profile.json'
AGE_REFERENCES_PATH = DRIVE_PROJECT / 'e1_age_references.json'
E1_REFERENCE_PATH.write_text(json.dumps(E1_REFERENCE_PROFILE, ensure_ascii=False), encoding='utf-8')
AGE_REFERENCES_PATH.write_text(json.dumps(AGE_REFERENCES, ensure_ascii=False), encoding='utf-8')
run_stage(
    '--stage', 'e1',
    '--reference-profile', str(E1_REFERENCE_PATH),
    '--age-references', str(AGE_REFERENCES_PATH),
    '--gestante', 'sim' if GESTANTE_PERFIS else 'nao',
)
print('Artefato E1:', MODELS_DIR / 'e1_final.joblib')

## 9. Abrir a demo no Colab

Execute após treinar E1 e M1. A demo usa somente os artefatos salvos no Drive e não carrega microdados. Não publique o app sem autorização explícita.

In [ ]:
import os
import time

from google.colab import output

if (
    not (MODELS_DIR / 'e1_final.joblib').is_file()
    or not (MODELS_DIR / 'm1_final.joblib').is_file()
):
    raise FileNotFoundError('Treine E1 e M1 antes de abrir a demo.')

env = os.environ.copy()
env['DENGUE_MODELS_DIR'] = str(MODELS_DIR)
server_log = (DRIVE_PROJECT / 'streamlit.log').open('w', encoding='utf-8')
streamlit_process = subprocess.Popen(
    [
        'uv', 'run', '--python', '3.13.12', 'streamlit', 'run',
        'app/streamlit_app.py', '--server.headless', 'true', '--server.port', '8501',
    ],
    cwd=WORKSPACE,
    env=env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
if streamlit_process.poll() is not None:
    server_log.close()
    raise RuntimeError((DRIVE_PROJECT / 'streamlit.log').read_text(encoding='utf-8'))
output.serve_kernel_port_as_window(8501)

## Encerrar a sessão

Confirme que as saídas foram gravadas no Drive antes de desconectar. Não execute avaliação em 2024 neste notebook de treinamento. Depois do congelamento e da revisão dos artefatos, a avaliação de teste deve ser acionada separadamente por `dengue_sinan.evaluate.evaluate_test_once`, apenas uma vez.